# BTCUSDT London Volatility Breakout
## Five-year Python research study

**Research question:** Can a dynamic breakout of the previous rolling hour capture directional BTCUSDT volatility during the London morning while keeping loss severity stable through equity-based position sizing?

This notebook presents the hypothesis, data controls, execution assumptions, performance and limitations.

## 1. Hypothesis and fixed rules

- Test period: 1 January 2021 through 31 December 2025.
- During 08:00–11:00 London, update stops at the previous 60 completed one-minute bars' high and low.
- Permit one trade per London day and cancel the opposite direction after entry.
- Risk 0.05% of current equity at a 0.5%-of-entry initial stop.
- Trail by the same distance, use no take-profit, and force-close at 16:00 London.
- Skip a candle when both entry levels trade and the sequence is unknowable.

## 2. Data and controls

The study uses public Binance BTCUSDT spot one-minute archives. The downloader checks the published SHA-256 checksum for every monthly ZIP. The loader enforces ordered UTC timestamps, rejects conflicting duplicates and validates every OHLC row. Raw archives are excluded from GitHub and can be downloaded reproducibly.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
trades = pd.read_csv(ROOT / 'results' / 'trades.csv', parse_dates=['entry_time_utc', 'exit_time_utc'])
equity = pd.read_csv(ROOT / 'results' / 'daily_equity.csv', index_col=0, parse_dates=True)['equity_usd']
yearly = pd.read_csv(ROOT / 'results' / 'yearly_results.csv', index_col='year')
summary = json.loads((ROOT / 'results' / 'summary.json').read_text())
summary['data_quality']

## 3. Backtest architecture

Each London trading day is processed chronologically. Entry levels use only completed earlier candles. Stops are checked before a completed bar can tighten the trail. Adverse gaps fill at the worse of the opening price and stop price. These rules reduce look-ahead bias but cannot recover the unknown sequence inside a one-minute candle.

## 4. Performance

| Metric | Result |
|---|---:|
| Ending equity | $153,944.59 |
| Total return | 2.63% |
| CAGR | 0.52% |
| Maximum drawdown | -1.34% |
| Sharpe ratio | 0.62 |
| Trades | 1,301 |
| Win rate | 39.35% |
| Profit factor | 1.11 |

![Equity and drawdown](../results/equity_drawdown.png)

In [ ]:
yearly[['trades', 'pnl_usd', 'win_rate']]

## 5. Equity-based scaling

`risk budget = current equity × 0.0005`

`quantity = risk budget ÷ stop distance`

The risk budget compounds with equity. BTC quantity can decline when price rises because the 0.5% dollar stop becomes larger.

![Risk scaling](../results/risk_position_scaling.png)

In [ ]:
trades[['initial_risk_usd', 'units', 'pnl_usd', 'ending_equity_usd']].describe()

## 6. Findings and limitations

The rule produced a small positive result before costs, with most gains concentrated in 2024–2025. Increasing risk from 0.02% to 0.05% scaled both return and drawdown but did not improve the profit factor or Sharpe ratio.

Fees, spread, slippage and financing are set to zero. One-minute bars do not reveal intraminute sequencing. The result therefore demonstrates a research process—not a production-ready trading system or forecast. A stronger next study would add realistic costs and preserve an untouched out-of-sample period.